# Ingerindo um arquivo CSV

Outro exemplo, em que queremos desserializar **e** validar dados — lendo de um arquivo CSV.

Arquivos CSV não têm tipos de dados: tudo é string. Então usamos o Pydantic também para coagir/converter os dados conforme a aplicação precisa.

Nesta pasta há um CSV de exemplo: [`pop_estimates.csv`](pop_estimates.csv).

Inspecionando o arquivo:

In [1]:
import csv

with open("pop_estimates.csv") as f:
    data = csv.reader(f)
    for _ in range (5):
        print(f"next(data): {next(data)}")

next(data): ['Geographic Area', 'July 1, 2001 Estimate', 'July 1, 2000 Estimate', 'April 1, 2000 Population Estimates Base']
next(data): ['United States', ' 284,796,887 ', ' 282,124,631 ', ' 281,421,906 ']
next(data): ['Alabama', ' 4,464,356 ', ' 4,451,493 ', ' 4,447,100 ']
next(data): ['Alaska', ' 634,892 ', ' 627,601 ', ' 626,932 ']
next(data): ['Arizona', ' 5,307,331 ', ' 5,165,274 ', ' 5,130,632 ']


A primeira linha tem os nomes das colunas, mas as estimativas de população estão assim:
```
' 284,796,887 '
```

O Pydantic é bom, mas não **tanto**! Vamos precisar de um validador customizado que converta valores como esse em inteiros na desserialização.

> 📝 **Nota de revisão:** a mensagem de erro original diz *"data could be parsed into a valid integer"* — faltou o **not**. Corrigido abaixo.

In [2]:
def name_int(value: str):
    try:
        return int(value.strip().replace(",", ""))
    except Exception as ex:
        raise ValueError("data could not be parsed into a valid integer")      # corrigido: faltava o "not"

Vamos criar um tipo `Annotated` que incorpora esse conversor. Aqui o validador precisa ser **before**: queremos que o campo seja `int`, e com um validador **after** teríamos problema, porque o Pydantic não consegue converter esses valores em `int` sozinho.

In [3]:
from typing import Annotated

from pydantic import BeforeValidator


FunkyInt = Annotated[int, BeforeValidator(name_int)]

Agora o modelo:

In [4]:
from pydantic import BaseModel

In [5]:
class Estimate(BaseModel):
    area: str
    july_1_2001: FunkyInt
    july_1_2000: FunkyInt
    april_1_2000: FunkyInt

Vamos ler o CSV com `csv.DictReader` — a vantagem é que cada linha vira um dicionário, com os nomes dos campos como chaves, fácil de passar para `model_validate()`.

Vejamos como funciona antes de usar o modelo:

In [6]:
with open("pop_estimates.csv") as f:
    data = csv.DictReader(f, fieldnames=["area", "july_1_2001", "july_1_2000", "april_1_2000"])
    for _ in range (5):
        print(f"next(data): {next(data)}")

next(data): {'area': 'Geographic Area', 'july_1_2001': 'July 1, 2001 Estimate', 'july_1_2000': 'July 1, 2000 Estimate', 'april_1_2000': 'April 1, 2000 Population Estimates Base'}
next(data): {'area': 'United States', 'july_1_2001': ' 284,796,887 ', 'july_1_2000': ' 282,124,631 ', 'april_1_2000': ' 281,421,906 '}
next(data): {'area': 'Alabama', 'july_1_2001': ' 4,464,356 ', 'july_1_2000': ' 4,451,493 ', 'april_1_2000': ' 4,447,100 '}
next(data): {'area': 'Alaska', 'july_1_2001': ' 634,892 ', 'july_1_2000': ' 627,601 ', 'april_1_2000': ' 626,932 '}
next(data): {'area': 'Arizona', 'july_1_2001': ' 5,307,331 ', 'july_1_2000': ' 5,165,274 ', 'april_1_2000': ' 5,130,632 '}


Só precisamos pular a primeira linha, que tem os cabeçalhos (que não precisamos nem usamos).

Também criamos uma função geradora, que encapsula a leitura dos dados.

In [7]:
def estimates():
    with open("pop_estimates.csv") as f:
        data = csv.DictReader(f, fieldnames=["area", "july_1_2001", "july_1_2000", "april_1_2000"])
        next(data)  # skip header row

        for row in data:
            yield Estimate.model_validate(row)

Podemos simplesmente iterar pelos dados:

In [8]:
for estimate in estimates():
    print(f"estimate: {estimate}")

estimate: area='United States' july_1_2001=284796887 july_1_2000=282124631 april_1_2000=281421906
estimate: area='Alabama' july_1_2001=4464356 july_1_2000=4451493 april_1_2000=4447100
estimate: area='Alaska' july_1_2001=634892 july_1_2000=627601 april_1_2000=626932
estimate: area='Arizona' july_1_2001=5307331 july_1_2000=5165274 april_1_2000=5130632
estimate: area='Arkansas' july_1_2001=2692090 july_1_2000=2678030 april_1_2000=2673400
estimate: area='California' july_1_2001=34501130 july_1_2000=34000446 april_1_2000=33871648
estimate: area='Colorado' july_1_2001=4417714 july_1_2000=4323410 april_1_2000=4301261
estimate: area='Connecticut' july_1_2001=3425074 july_1_2000=3410079 april_1_2000=3405565
estimate: area='Delaware' july_1_2001=796165 july_1_2000=786234 april_1_2000=783600
estimate: area='District of Columbia' july_1_2001=571822 july_1_2000=571066 april_1_2000=572059
estimate: area='Florida' july_1_2001=16396515 july_1_2000=16054328 april_1_2000=15982378
estimate: area='Georgia

ou transformá-los em lista e trabalhar com elementos individuais:

In [9]:
data = list(estimates())

In [10]:
print(f"data[0]: {data[0]!r}")

data[0]: Estimate(area='United States', july_1_2001=284796887, july_1_2000=282124631, april_1_2000=281421906)


> 📝 **Nota de revisão:** a documentação do módulo `csv` recomenda abrir o arquivo com `newline=""` (para campos com quebra de linha entre aspas serem lidos corretamente). Não faz diferença neste arquivo, mas é o padrão a adotar.

> 💡 **Complemento — erros com número de linha:** do jeito acima, a primeira linha inválida interrompe a geração com um `ValidationError` que não diz **qual linha** do arquivo falhou. Numa ingestão real, é comum registrar o erro com a linha e seguir em frente:

In [11]:
import io
from pydantic import ValidationError

FIELDNAMES = ["area", "july_1_2001", "july_1_2000", "april_1_2000"]


def estimates_with_errors(f):
    reader = csv.DictReader(f, fieldnames=FIELDNAMES)
    next(reader)                                        # pula o cabeçalho
    for row in reader:
        try:
            yield Estimate.model_validate(row), None
        except ValidationError as ex:
            yield None, (reader.line_num, [(e["loc"], e["msg"]) for e in ex.errors()])


bad_csv = io.StringIO(
    'Geographic Area,"July 1, 2001","July 1, 2000","April 1, 2000"\n'
    'Alabama," 4,464,356 "," 4,451,493 "," 4,447,100 "\n'
    'Atlantis," n/a "," 1,000 "," 1,000 "\n'
    'Alaska," 634,892 "," 627,601 "," 626,932 "\n'
)
for estimate, error in estimates_with_errors(bad_csv):
    if error:
        print(f"linha {error[0]} rejeitada: {error[1]}")
    else:
        print(f"ok: {estimate.area}")

ok: Alabama
linha 3 rejeitada: [(('july_1_2001',), 'Value error, data could not be parsed into a valid integer')]
ok: Alaska
